In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

I0000 00:00:1785135168.250452  185129 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785135168.305993  185129 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785135170.171797  185129 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
rom tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

# Base Model
base_model = EfficientNetB0(

    weights="imagenet",

    include_top=False,

    input_shape=(128,128,3)

)

base_model.trainable = False

# Build Model
efficientnet = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(256, activation="relu"),

    Dropout(0.5),

    Dense(NUM_CLASSES, activation="softmax")

])

efficientnet.summary()

In [2]:
#Load Metadata
metadata = pd.read_csv("Datasets/HAM10000_metadata.csv")

In [3]:
#Create Image Path
import glob

#Read Images
image_paths = {}

folders = [
    "Datasets/HAM10000_images_part_1",
    "Datasets/HAM10000_images_part_2"
]

for folder in folders:

    
    search_pattern = os.path.join(folder, "*.jpg")

   
    all_images = glob.glob(search_pattern)

    
    for image_path in all_images:

        
        file_name = os.path.basename(image_path)

        
        image_id = os.path.splitext(file_name)[0]

        
        image_paths[image_id] = image_path

metadata["path"] = metadata["image_id"].map(image_paths)

In [4]:
#Encode Target Labels
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

metadata["label"] = encoder.fit_transform(metadata["dx"])

metadata[["dx","label"]].head()


,dx,label
0,bkl,2
1,bkl,2
2,bkl,2
3,bkl,2
4,bkl,2


In [5]:
#Train / Validation / Test Split
train_df, temp_df = train_test_split(
    metadata,
    test_size=0.30,
    stratify=metadata["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

In [6]:
#Create Image Loader
IMG_SIZE = 224

def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))

    image = tf.cast(image, tf.float32)

    image = image / 255.0

    return image, label

In [7]:
#Create TensorFlow Datasets
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"].values,
        train_df["label"].values
    )
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (
        val_df["path"].values,
        val_df["label"].values
    )
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"].values,
        test_df["label"].values
    )
)

E0000 00:00:1785135172.319491  185129 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
#Map Image Loader
train_ds = train_ds.map(load_image)

val_ds = val_ds.map(load_image)

test_ds = test_ds.map(load_image)

In [9]:
#Data Augmentation
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip(
        "horizontal_and_vertical"
    ),

    tf.keras.layers.RandomRotation(
        0.2
    ),

    tf.keras.layers.RandomZoom(
        0.2
    ),

    tf.keras.layers.RandomContrast(
        0.2
    ),

    tf.keras.layers.RandomBrightness(
        factor=0.2
    ),

    tf.keras.layers.RandomCrop(
        200,
        200
    ),

    tf.keras.layers.Resizing(
        224,
        224
    )

])

In [10]:
#Apply Augmentation Only to Training Data
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)

In [11]:
#Handle Class Imbalance
train_df["label"].value_counts()

label
5    4693
4     779
2     769
1     360
0     229
6      99
3      81
Name: count, dtype: int64

In [12]:
#Compute Class Weights

unique_classes = np.unique(train_df["label"])

print("Unique Classes:")
print(unique_classes)

training_labels = train_df["label"]

weights = compute_class_weight(
    class_weight="balanced",
    classes=unique_classes,
    y=training_labels
)

print("\nCalculated Weights:")
print(weights)

class_weights = {}

# Store each class and its weight
for i in range(len(unique_classes)):

    class_number = unique_classes[i]

    weight = weights[i]

    class_weights[class_number] = weight


print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


In [13]:
#Batch and Prefetch
BATCH_SIZE = 32

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)